In [1]:
# ViT-Tiny with frozen backbone -- only train the classification head (193 params).
# No augmentation. Test on 0/90/180/270.

import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

DATA_DIR = r"E:\Deep Learning Spring 26\dataset"
BATCH_SIZE = 32
EPOCHS = 6
LR = 1e-3
NUM_WORKERS = 0
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
train_dir = os.path.join(DATA_DIR, "train")
df = pd.read_csv(os.path.join(DATA_DIR, "train_labels.csv"))

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df.label, random_state=SEED)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

Train: 176020  |  Val: 22002  |  Test: 22003


In [3]:
# no augmentation, no resize (timm handles 96x96 via img_size param)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CancerDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, rotation=0):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.rotation = rotation
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row.id + ".tif")).convert("RGB")
        if self.rotation != 0:
            img = img.rotate(self.rotation)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row.label, dtype=torch.float32)

In [4]:
train_loader = DataLoader(CancerDataset(train_df, train_dir, transform, rotation=0),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(CancerDataset(val_df, train_dir, transform, rotation=0),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

test_loaders = {}
for angle in [0, 90, 180, 270]:
    test_loaders[angle] = DataLoader(
        CancerDataset(test_df, train_dir, transform, rotation=angle),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [5]:
# ViT-Tiny: 5.7M params, 12 layers, 3 heads, 192-dim embeddings
# img_size=96 gives us 6x6=36 patches instead of 14x14=196
model = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=1, img_size=96)

# freeze everything except the head
for name, param in model.named_parameters():
    if "head" not in name:
        param.requires_grad = False
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {num_params:,}")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.head.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

Trainable params: 193


In [6]:
def train_one_epoch():
    model.train()
    total_loss = 0
    preds_list, labels_list = [], []
    for imgs, labels in tqdm(train_loader, desc="Training", leave=False):
        imgs = imgs.to(device)
        labels = labels.unsqueeze(1).to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds_list.extend(torch.sigmoid(out).squeeze().detach().cpu().numpy())
        labels_list.extend(labels.squeeze().cpu().numpy())
    avg_loss = total_loss / len(train_loader)
    p, l = np.array(preds_list), np.array(labels_list).astype(int)
    b = (p >= 0.5).astype(int)
    return avg_loss, accuracy_score(l, b), f1_score(l, b, average="weighted"), roc_auc_score(l, p)

def evaluate(loader):
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds_list.extend(torch.sigmoid(model(imgs)).squeeze().cpu().numpy())
            labels_list.extend(labels.numpy())
    p, l = np.array(preds_list), np.array(labels_list).astype(int)
    b = (p >= 0.5).astype(int)
    return accuracy_score(l, b), f1_score(l, b, average="weighted"), roc_auc_score(l, p)

In [7]:
best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    loss, train_acc, train_f1, train_auc = train_one_epoch()
    scheduler.step()
    val_acc, val_f1, val_auc = evaluate(val_loader)

    lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch}/{EPOCHS} (lr={lr:.2e})")
    print(f"  Train  =>  Loss: {loss:.4f}  Acc: {train_acc:.4f}  F1: {train_f1:.4f}  AUC: {train_auc:.4f}")
    print(f"  Val    =>  Acc: {val_acc:.4f}  F1: {val_f1:.4f}  AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_vit.pth")
        print(f"  >> Saved best model (AUC={val_auc:.4f})")


--- Epoch 1/6 (lr=9.33e-04) ---
  Train  =>  Loss: 0.3933  Acc: 0.8239  F1: 0.8229  AUC: 0.8976
  Val    =>  Acc: 0.8273  F1: 0.8239  AUC: 0.9057
  >> Saved new best model (AUC=0.9057)



--- Epoch 2/6 (lr=7.50e-04) ---
  Train  =>  Loss: 0.3852  Acc: 0.8290  F1: 0.8282  AUC: 0.9022
  Val    =>  Acc: 0.8252  F1: 0.8261  AUC: 0.9077
  >> Saved new best model (AUC=0.9077)



--- Epoch 3/6 (lr=5.00e-04) ---
  Train  =>  Loss: 0.3826  Acc: 0.8305  F1: 0.8297  AUC: 0.9036
  Val    =>  Acc: 0.8325  F1: 0.8304  AUC: 0.9081
  >> Saved new best model (AUC=0.9081)



--- Epoch 4/6 (lr=2.50e-04) ---
  Train  =>  Loss: 0.3797  Acc: 0.8315  F1: 0.8307  AUC: 0.9051
  Val    =>  Acc: 0.8324  F1: 0.8314  AUC: 0.9083
  >> Saved new best model (AUC=0.9083)



--- Epoch 5/6 (lr=6.70e-05) ---
  Train  =>  Loss: 0.3772  Acc: 0.8326  F1: 0.8317  AUC: 0.9063
  Val    =>  Acc: 0.8331  F1: 0.8303  AUC: 0.9087
  >> Saved new best model (AUC=0.9087)



--- Epoch 6/6 (lr=0.00e+00) ---
  Train  =>  Loss: 0.3751  Acc: 0.8337  F1: 0.8329  AUC: 0.9074
  Val    =>  Acc: 0.8347  F1: 0.8340  AUC: 0.9089
  >> Saved new best model (AUC=0.9089)

ROTATION TEST (best model)


In [8]:
print("\nRotation test (best model):")
model.load_state_dict(torch.load("best_vit.pth"))
for angle in [0, 90, 180, 270]:
    acc, f1, auc = evaluate(test_loaders[angle])
    print(f"  {angle:3d}°  =>  Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")

C:\Users\AhmedFahmy\AppData\Local\Temp\ipykernel_12084\1768442068.py:242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_vit.pth"))


    0°  =>  Acc: 0.8335  F1: 0.8329  AUC: 0.9082
   90°  =>  Acc: 0.8345  F1: 0.8340  AUC: 0.9084
  180°  =>  Acc: 0.8347  F1: 0.8341  AUC: 0.9073
  270°  =>  Acc: 0.8345  F1: 0.8340  AUC: 0.9083

Done!
